In [6]:
import os
import zipfile
import cv2
import numpy as np
from google.colab import files
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

uploaded = files.upload()
zip_file = list(uploaded.keys())[0]
print("Uploaded ZIP File:",zip_file)

# Ensure the extraction directory exists
extract_dir = '/content/gesture_data'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
  zip_ref.extractall(extract_dir)
print("ZIP file extracted successfully.")

# Find the actual directory containing images
# Assumption: The zip file extracts into a single subdirectory within extract_dir
# e.g., /content/gesture_data/hand_gesture/
image_subdirs = [d for d in os.listdir(extract_dir) if os.path.isdir(os.path.join(extract_dir, d))]

image_data_root = None
if len(image_subdirs) == 1:
    image_data_root = os.path.join(extract_dir, image_subdirs[0])
elif not image_subdirs and any(f.endswith(('.jpg', '.png')) for f in os.listdir(extract_dir)):
    # If no subdirs, but images are directly in extract_dir
    image_data_root = extract_dir
else:
    print(f"Warning: Unexpected directory structure in {extract_dir}. Found: {os.listdir(extract_dir)}")
    # Fallback if `image_subdirs` is empty but images might be directly in `extract_dir`
    image_data_root = extract_dir

if image_data_root is None or not os.path.isdir(image_data_root):
    # This case should ideally not be reached with the current logic, but as a safeguard
    print(f"Error: Could not determine valid image data root. Current path: {image_data_root}")
    raise FileNotFoundError("Could not find a valid directory to load images from.")


print(f"Loading images from: {image_data_root}")

data = []
labels = []

# List files from the determined image_data_root
for image_filename in os.listdir(image_data_root):
  if image_filename.endswith('.jpg') or image_filename.endswith('.png'):
    img_path = os.path.join(image_data_root, image_filename) # Corrected path construction
    img = cv2.imread(img_path)
    if img is None:
      print(f"Skipping corrupt or unreadable image: {image_filename}")
      continue
    # Fix: cv2.resize expects (width, height) tuple
    img = cv2.resize(img, (64, 64))
    img = img.flatten()
    data.append(img)
    if 'thumb' in image_filename.lower():
      labels.append(0)
    elif 'palm' in image_filename.lower():
      labels.append(1)
    elif 'victory' in image_filename.lower():
      labels.append(2)

if not data:
    print("Error: No images were loaded. Check the zip file content and extraction path.")
    # Raise an error to prevent the ValueError in train_test_split
    raise ValueError("No images loaded, cannot train model.")

X = np.array(data)
y = np.array(labels)

# Add a check for empty data after converting to numpy arrays
if X.shape[0] == 0:
    raise ValueError("No samples found in X after image loading. Cannot proceed with train_test_split.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = SVC()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Model Trained Successfully.")
print("Accuracy:", accuracy)

Saving hand_gesture.zip to hand_gesture (3).zip
Uploaded ZIP File: hand_gesture (3).zip
ZIP file extracted successfully.
Loading images from: /content/gesture_data/hand_gesture
Model Trained Successfully.
Accuracy: 1.0
